In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import List, Tuple, Literal

coverage = pd.read_csv("coverage.tsv", sep="\t")

In [ ]:
def extract_id(name: str) -> str:
    """
    Extracts the sample ID from the column name.
    Parameters
    name : str
        Column name, for example: "sample_01sib_normal" or "control"

    Returns
    str
        Extracted identifier (for example, "01sib") or the original name,
        if the pattern is not found.
    """
    m = re.search(r"_(\d+sib)_", name)
    return m.group(1) if m else name


def reorder_samples(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Rearranges the sample columns in the DataFrame by the numeric identifier 'sib'.

    The function highlights meta columns ("chr", "start", "genome_pos") and sample columns.
    For each column of the sample, it extracts the ID of the form "<number>sib", sorts them by number,
    and returns a DataFrame with columns in order: meta columns + sorted samples.
    Parameters
    df : pd.DataFrame
        The input DataFrame, which should contain the columns "chr", "start", "genome_pos"

    Returns
        Tuple[pd.DataFrame, List[str]]
        A tuple of two elements:
        - df_reordered : pd.DataFrame
            A DataFrame with reordered columns
        - ordered_sibs : List[str]
            A list of sample IDs in sorted order (for example, ["1sib", "2sib"])
    """
    meta_cols = ["chr", "start", "genome_pos"]
    sample_cols = [c for c in df.columns if c not in meta_cols]

    name_map = {}
    for c in sample_cols:
        sid = extract_id(c)
        if sid:
            name_map[sid] = c

    def sib_sort(x):
        return int(x.replace("sib", ""))

    ordered_sibs = sorted(name_map.keys(), key=sib_sort)

    ordered_cols = [name_map[s] for s in ordered_sibs]

    return df[meta_cols + ordered_cols], ordered_sibs

In [ ]:
def plot_corr_heatmap(
    df: pd.DataFrame,
    use: Literal["norm", "raw"] = "norm",
    out_file: str = "corr_heatmap_final.png",
) -> None:
    """
    Plot correlation heatmap for sample columns.

    Parameters
    df : pd.DataFrame
        Input data with meta columns ["chr", "start", "genome_pos"] and sample columns.
    use : "norm" or "raw", default "norm"
        - "norm": normalize by autosomal mean before correlation
        - "raw": use raw values
    out_file : str, default "corr_heatmap_final.png"
        Output file path for the saved heatmap.
    Returns
    None
        Saves plot to file and closes it.
    """

    meta_cols = ["chr", "start", "genome_pos"]
    sample_cols = [c for c in df.columns if c not in meta_cols]

    numeric = df[sample_cols].astype(float)

    if use == "norm":
        auto_mask = ~df["chr"].isin(["chrX", "X", "chrY", "Y"])
        auto_mean = numeric.loc[auto_mask].mean(axis=0)
        data = numeric.div(auto_mean, axis=1)
    else:
        data = numeric

    corr = data.corr()

    new_names = {c: extract_id(c) for c in corr.columns}
    corr.rename(index=new_names, columns=new_names, inplace=True)

    plt.figure(figsize=(10, 8))

    sns.heatmap(
        corr,
        cmap="coolwarm",
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        square=True,
        linewidths=0.5,
    )

    plt.title("Sample correlation")
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(out_file, dpi=150)
    plt.close()

In [4]:
plot_corr_heatmap(coverage)